In [1]:
import rclpy
from rclpy.node import Node
import json
import sys

# Initialize ROS2
rclpy.init()

# Create a test node
test_node = Node('assembly_knowledge_tools_test_node')
print("✓ ROS2 node created successfully")

✓ ROS2 node created successfully


In [2]:
# Import AssemblyKnowledgeTools - workaround for package naming conflicts
import sys
import os
from pathlib import Path
import types

# Get the notebook directory
notebook_dir = Path(__file__).parent if '__file__' in dir() else Path.cwd()
package_root = notebook_dir.parent.parent

# Create module hierarchy to avoid naming conflicts
pm_co_pilot_planning = types.ModuleType('pm_co_pilot_planning')
submodules = types.ModuleType('submodules')
langchain = types.ModuleType('langchain')
tools = types.ModuleType('tools')

# Set up the package hierarchy
pm_co_pilot_planning.submodules = submodules
submodules.langchain = langchain
langchain.tools = tools

sys.modules['pm_co_pilot_planning'] = pm_co_pilot_planning
sys.modules['pm_co_pilot_planning.submodules'] = submodules
sys.modules['pm_co_pilot_planning.submodules.langchain'] = langchain
sys.modules['pm_co_pilot_planning.submodules.langchain.tools'] = tools

# Add paths
sys.path.insert(0, str(notebook_dir))
sys.path.insert(0, str(package_root))

# Now import the actual modules
try:
    # Import _helpers first
    import importlib.util
    helpers_spec = importlib.util.spec_from_file_location(
        "pm_co_pilot_planning.submodules.langchain.tools._helpers",
        str(notebook_dir / "submodules" / "langchain" / "tools" / "_helpers.py")
    )
    helpers_module = importlib.util.module_from_spec(helpers_spec)
    sys.modules["pm_co_pilot_planning.submodules.langchain.tools._helpers"] = helpers_module
    helpers_spec.loader.exec_module(helpers_module)
    
    # Import AssemblyKnowledgeTools
    akt_spec = importlib.util.spec_from_file_location(
        "pm_co_pilot_planning.submodules.langchain.tools.AssemblyKnowledgeTools",
        str(notebook_dir / "submodules" / "langchain" / "tools" / "AssemblyKnowledgeTools.py")
    )
    akt_module = importlib.util.module_from_spec(akt_spec)
    sys.modules["pm_co_pilot_planning.submodules.langchain.tools.AssemblyKnowledgeTools"] = akt_module
    akt_spec.loader.exec_module(akt_module)
    
    AssemblyKnowledgeTools = akt_module.AssemblyKnowledgeTools
    print("✓ AssemblyKnowledgeTools imported successfully")
except Exception as e:
    print(f"✗ Import error: {e}")
    import traceback
    traceback.print_exc()

✓ AssemblyKnowledgeTools imported successfully


In [3]:
# Initialize AssemblyKnowledgeTools
try:
    assembly_tools = AssemblyKnowledgeTools(service_node=test_node)
    print("✓ AssemblyKnowledgeTools initialized successfully")
except Exception as e:
    print(f"✗ Failed to initialize AssemblyKnowledgeTools: {e}")

[INFO] [1774883733.627881584] [assembly_knowledge_tools_test_node]: Initializing AssemblyKnowledgeTools...


✓ AssemblyKnowledgeTools initialized successfully


[INFO] [1774883734.449129032] [assembly_knowledge_tools_test_node]: Config loaded from file: /home/match-pm/ros2_ws/install/ros_sequential_action_programmer/share/ros_sequential_action_programmer/rsap_config.yaml
[INFO] [1774883734.454526032] [assembly_knowledge_tools_test_node]: Subscribed to /assembly_manager/scene for live scene updates.


In [10]:
# Test: List available components
print("\n=== Test 1: List Available Components ===")
try:
    result = assembly_tools._list_available_components()
    print(f"Result:\n{result}")
    components_data = json.loads(result)
    if components_data.get('success'):
        print(f"\n✓ Found {components_data.get('count', 0)} components")
        for comp in components_data.get('components', [])[:5]:  # Show first 5
            print(f"  - {comp['name']} (gonio: {comp['gonio_side']})")
        if len(components_data.get('components', [])) > 5:
            print(f"  ... and {len(components_data.get('components', [])) - 5} more")
    else:
        print(f"✗ Error: {components_data}")
except Exception as e:
    print(f"✗ Exception: {e}")


=== Test 1: List Available Components ===
Result:
{"success": true, "count": 6, "components": [{"name": "Glas_Platelet_Paper", "file_path": "/home/match-pm/Documents/co_pilot_planning_test/Assembly_Part_Data/Co_Pilot_Planning_Tests/Sensor_Dummy/components/Glas_Platelet_Paper.json", "gonio_side": "left"}, {"name": "Glas_Platelet_Paper_ideal", "file_path": "/home/match-pm/Documents/co_pilot_planning_test/Assembly_Part_Data/Co_Pilot_Planning_Tests/Sensor_Dummy/components/Glas_Platelet_Paper_ideal.json", "gonio_side": "left"}, {"name": "Light_Sensor", "file_path": "/home/match-pm/Documents/co_pilot_planning_test/Assembly_Part_Data/Co_Pilot_Planning_Tests/Carrier_Dummy_Assembly/components/Light_Sensor.json", "gonio_side": "right"}, {"name": "Sensor_Holder", "file_path": "/home/match-pm/Documents/co_pilot_planning_test/Assembly_Part_Data/Co_Pilot_Planning_Tests/Carrier_Dummy_Assembly/components/Sensor_Holder.json", "gonio_side": "left"}, {"name": "UFC_Paper", "file_path": "/home/match-pm/Do

In [13]:
# Test: Get component description
print("\n=== Test 2: Get Component Description ===")
try:
    # Try with first component name from previous list (if available)
    result = assembly_tools._list_available_components()
    components_data = json.loads(result)
    
    if components_data.get('components'):
        first_component_name = components_data['components'][2]['name']
        print(f"Testing with component: {first_component_name}")
        
        result = assembly_tools._get_component_description(first_component_name)
        comp_desc = json.loads(result)
        
        if comp_desc.get('success'):
            print(f"✓ Component description loaded successfully")
            print(f"  Name: {comp_desc.get('name')}")
            print(f"  Gonio side: {comp_desc.get('gonio_side')}")
            print(f"  Vision points: {comp_desc.get('frames', {}).get('vision_points', [])}")
            print(f"  Laser frames: {comp_desc.get('frames', {}).get('laser_measurement_frames', [])}")
            print(f"  Gripping point: {comp_desc.get('frames', {}).get('gripping_point', 'N/A')}")
            print(f"  Glue points: {len(comp_desc.get('frames', {}).get('glue_points', []))} found")
        else:
            print(f"✗ Error: {comp_desc}")
    else:
        print("ℹ No components found in database")
except Exception as e:
    print(f"✗ Exception: {e}")


=== Test 2: Get Component Description ===
Testing with component: Light_Sensor
✓ Component description loaded successfully
  Name: Light_Sensor
  Gonio side: right
  Vision points: ['Vision_1', 'Vision_2', 'Vision_3', 'Vision_4']
  Laser frames: []
  Gripping point: Gripping_Frame
  Glue points: 0 found


In [6]:
# Test: List available assemblies
print("\n=== Test 3: List Available Assemblies ===")
try:
    result = assembly_tools._list_available_assemblies()
    assemblies_data = json.loads(result)
    if assemblies_data.get('success'):
        print(f"✓ Found {assemblies_data.get('count', 0)} assemblies")
        for asm in assemblies_data.get('assemblies', [])[:5]:  # Show first 5
            print(f"  - {asm['name']} (components: {', '.join(asm['components']) if asm['components'] else 'none'})")
        if len(assemblies_data.get('assemblies', [])) > 5:
            print(f"  ... and {len(assemblies_data.get('assemblies', [])) - 5} more")
    else:
        print(f"✗ Error: {assemblies_data}")
except Exception as e:
    print(f"✗ Exception: {e}")


=== Test 3: List Available Assemblies ===
✓ Found 3 assemblies
  - Glas_UFC (components: Glas_Platelet_Paper, UFC_Paper)
  - Ideale_Baugruppe (components: Glas_Platelet_Paper_ideal, UFC_Paper_ideal)
  - Sensor_System_Assembly (components: Light_Sensor-1, Light_Sensor-2, Sensor_Holder-1)


In [4]:
# Test: Get assembly description
print("\n=== Test 4: Get Assembly Description ===")
try:
    result = assembly_tools._list_available_assemblies()
    assemblies_data = json.loads(result)
    
    if assemblies_data.get('assemblies'):
        first_assembly_name = assemblies_data['assemblies'][2]['name']
        print(f"Testing with assembly: {first_assembly_name}")
        
        result = assembly_tools._get_assembly_description(first_assembly_name)
        asm_desc = json.loads(result)
        print(f"Result:\n{result}")
        if asm_desc.get('success'):
            print(f"✓ Assembly description loaded successfully")
            print(f"  Name: {asm_desc.get('name')}")
            print(f"  Components: {len(asm_desc.get('components', []))} found")
            for comp in asm_desc.get('components', [])[:5]:
                print(f"    - {comp['name']}")
            print(f"  Constraints: {len(asm_desc.get('assembly_constraints', []))} found")
        else:
            print(f"✗ Error: {asm_desc}")
    else:
        print("ℹ No assemblies found in database")
except Exception as e:
    print(f"✗ Exception: {e}")


=== Test 4: Get Assembly Description ===
Testing with assembly: Sensor_System_Assembly
Result:
{"success": true, "name": "Sensor_System_Assembly", "file_path": "/home/match-pm/Documents/co_pilot_planning_test/Assembly_Part_Data/Co_Pilot_Planning_Tests/Carrier_Dummy_Assembly/assemblies/Sensor_System_Assembly.json", "components": [{"name": "Light_Sensor-1", "guid": ""}, {"name": "Light_Sensor-2", "guid": ""}, {"name": "Sensor_Holder-1", "guid": "324c3deb-6492-4f70-9b52-07fa6607a476"}], "assembly_constraints": [{"name": "Description_Sensor_Holder-1_Light_Sensor-1", "component_1": "Sensor_Holder-1", "component_2": "Light_Sensor-1", "plane_matches": 3}, {"name": "Description_Light_Sensor-2_Sensor_Holder-1", "component_1": "Light_Sensor-2", "component_2": "Sensor_Holder-1", "plane_matches": 3}], "assembly_frames": []}
✓ Assembly description loaded successfully
  Name: Sensor_System_Assembly
  Components: 3 found
    - Light_Sensor-1
    - Light_Sensor-2
    - Sensor_Holder-1
  Constraints: 

In [5]:
# Test: List objects in scene
print("\n=== Test 5: List Objects in Scene (Live) ===")
try:
    # Spin briefly to get latest scene data
    assembly_tools._ensure_scene_updated(timeout_sec=1.0)
    
    result = assembly_tools._list_objects_in_scene()
    scene_data = json.loads(result)
    
    if scene_data.get('success'):
        count = scene_data.get('count', 0)
        print(f"✓ Found {count} objects in scene")
        for obj in scene_data.get('objects', []):
            gripped = "✓" if obj['properties']['is_gripped'] else "✗"
            assembled = "✓" if obj['properties']['is_assembled'] else "✗"
            print(f"  - {obj['obj_name']} (parent: {obj['parent_frame']}, gripped: {gripped}, assembled: {assembled})")
        if count == 0:
            print("  (Scene is empty - no objects spawned)")
    else:
        print(f"✗ Error: {scene_data}")
except Exception as e:
    print(f"✗ Exception: {e}")


=== Test 5: List Objects in Scene (Live) ===
✗ Error: {'success': False, 'error': 'No scene received yet. The /assembly_manager/scene topic may not be publishing.'}


In [ ]:
# Test: Get object properties  
print("\n=== Test 6: Get Object Properties ===")
try:
    result = assembly_tools._list_objects_in_scene()
    scene_data = json.loads(result)
    
    if scene_data.get('objects'):
        first_obj_name = scene_data['objects'][0]['obj_name']
        print(f"Testing with object: {first_obj_name}")
        
        result = assembly_tools._get_object_properties(first_obj_name)
        obj_props = json.loads(result)
        
        if obj_props.get('success'):
            print(f"✓ Object properties loaded successfully")
            print(f"  Name: {obj_props.get('obj_name')}")
            print(f"  Parent frame: {obj_props.get('parent_frame')}")
            print(f"  Is gripped: {obj_props.get('properties', {}).get('is_gripped')}")
            print(f"  Is assembled: {obj_props.get('properties', {}).get('is_assembled')}")
        else:
            print(f"✗ Error: {obj_props}")
    else:
        print("ℹ No objects in scene to test")
except Exception as e:
    print(f"✗ Exception: {e}")

In [ ]:
# Test: Get object frames
print("\n=== Test 7: Get Object Frames ===")
try:
    result = assembly_tools._list_objects_in_scene()
    scene_data = json.loads(result)
    
    if scene_data.get('objects'):
        first_obj_name = scene_data['objects'][0]['obj_name']
        print(f"Testing with object: {first_obj_name}")
        
        result = assembly_tools._get_object_frames(first_obj_name)
        obj_frames = json.loads(result)
        
        if obj_frames.get('success'):
            frame_count = obj_frames.get('frame_count', 0)
            print(f"✓ Found {frame_count} frames for {first_obj_name}")
            for frame in obj_frames.get('frames', [])[:10]:  # Show first 10
                print(f"  - {frame.get('name', 'Unknown')} (parent: {frame.get('parent_frame', 'N/A')})")
            if frame_count > 10:
                print(f"  ... and {frame_count - 10} more frames")
        else:
            print(f"✗ Error: {obj_frames}")
    else:
        print("ℹ No objects in scene to test")
except Exception as e:
    print(f"✗ Exception: {e}")

In [ ]:
# Test: Get frame properties
print("\n=== Test 8: Get Frame Properties ===")
try:
    result = assembly_tools._list_objects_in_scene()
    scene_data = json.loads(result)
    
    if scene_data.get('objects'):
        first_obj_name = scene_data['objects'][0]['obj_name']
        
        result = assembly_tools._get_object_frames(first_obj_name)
        obj_frames = json.loads(result)
        
        if obj_frames.get('frames'):
            first_frame_name = obj_frames['frames'][0].get('name', '')
            print(f"Testing with frame: {first_frame_name}")
            
            result = assembly_tools._get_frame_properties(first_frame_name)
            frame_props = json.loads(result)
            
            if frame_props.get('success'):
                print(f"✓ Frame properties loaded successfully")
                print(f"  Name: {frame_props.get('name')}")
                print(f"  Parent frame: {frame_props.get('parent_frame')}")
                print(f"  Object: {frame_props.get('obj_name', 'N/A')}")
            else:
                print(f"✗ Error: {frame_props}")
        else:
            print("ℹ No frames found for first object")
    else:
        print("ℹ No objects in scene to test")
except Exception as e:
    print(f"✗ Exception: {e}")

In [ ]:
# Test: Get frames in scene
print("\n=== Test 9: Get Frames in Scene ===")
try:
    result = assembly_tools._get_frames_in_scene()
    frames_data = json.loads(result)
    
    if frames_data.get('success'):
        print(f"✓ Retrieved frame information")
        print(f"  Value sets count: {frames_data.get('count', 0)}")
        for set_name, set_info in list(frames_data.get('value_sets', {}).items())[:3]:
            print(f"  - {set_name}: {set_info.get('type')} ({len(set_info.get('values', []))} values)")
        if len(frames_data.get('value_sets', {})) > 3:
            print(f"  ... and {len(frames_data.get('value_sets', {})) - 3} more value sets")
    else:
        print(f"✗ Error: {frames_data}")
except Exception as e:
    print(f"✗ Exception: {e}")

In [ ]:
# Test: Get compact scene summary
print("\n=== Test 10: Get Compact Scene Summary ===")
try:
    summary = assembly_tools.get_compact_scene_summary()
    print("✓ Scene summary generated successfully:")
    print(summary)
except Exception as e:
    print(f"✗ Exception: {e}")

## Summary of Available Tools

The `AssemblyKnowledgeTools` provides the following tools for querying assembly database and live scene:

### Static Database Tools (Component & Assembly Information)
1. **list_available_components** - Scan database for all available component files
2. **get_component_description** - Get detailed component info (frames, types, properties)
3. **list_available_assemblies** - Scan database for all available assembly files
4. **get_assembly_description** - Get assembly info (components, constraints)

### Live Scene Tools (Real-time Assembly State)
5. **list_objects_in_scene** - Get all objects currently spawned in the scene
6. **get_object_properties** - Get properties of a specific object
7. **get_object_frames** - Get all reference frames attached to an object
8. **get_frame_properties** - Get detailed properties of a specific frame
9. **get_frames_in_scene** - Get all available frame value sets and TF frames

### Utility Methods
- **get_compact_scene_summary()** - Get a <300 token scene state summary for LLM context